# AI Rule Mining Discovery — VVMV

Mục tiêu notebook này là **khám phá quy luật mới phục vụ AI/autofill** từ dữ liệu declaration + cargo.

Notebook tập trung vào:
- Rule 2 chiều: `A -> B` như `operation_code -> direction`, `country -> currency`
- Rule 3 chiều: `A + B -> C` như `operation_code + export_country -> currency`
- HS hierarchy: `hs2/hs4/hs6 -> unit/tax_code/origin`
- Candidate rules theo ngưỡng AI: AUTOFILL / SUGGEST / HINT / ASK
- Xuất kết quả ra `notebooks/outputs/*.csv|json` để review hoặc đưa vào engine

In [1]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'notebooks' / 'lib'))

from ai_rule_mining import connect, register_views, q, dominant_rules, save, OUTPUT_DIR

pd.set_option('display.max_columns', 80)
pd.set_option('display.max_colwidth', 120)
sns.set_theme(style='whitegrid')
print(PROJECT_ROOT)
print(OUTPUT_DIR)

/Volumes/FastVault-Apps/Projects/ai-vvmv
/Volumes/FastVault-Apps/Projects/ai-vvmv/notebooks/outputs


## 1. Load dữ liệu bằng DuckDB
DuckDB đọc trực tiếp JSONL nên không cần load toàn bộ vào RAM Pandas.

In [2]:
con = connect(memory_limit='8GB', threads=8)
register_views(con)

summary = q(con, '''
SELECT 'declarations' AS table_name, COUNT(*) AS rows FROM declarations
UNION ALL
SELECT 'cargo' AS table_name, COUNT(*) AS rows FROM cargo
''')
summary

,table_name,rows
0,declarations,563919
1,cargo,2012434


## 2. Schema và độ phủ field
Trước khi mining rule, cần biết field nào đủ dữ liệu để tin cậy.

In [3]:
field_coverage = q(con, '''
WITH d AS (
  SELECT 'declarations' AS table_name, COUNT(*) AS n,
         SUM(direction IS NOT NULL AND direction <> '') AS direction,
         SUM(operation_code IS NOT NULL AND operation_code <> '') AS operation_code,
         SUM(export_country IS NOT NULL AND export_country <> '') AS export_country,
         SUM(import_country IS NOT NULL AND import_country <> '') AS import_country,
         SUM(delivery_term IS NOT NULL AND delivery_term <> '') AS delivery_term,
         SUM(currency IS NOT NULL AND currency <> '') AS currency,
         SUM(port_code IS NOT NULL AND port_code <> '') AS port_code,
         SUM(customs_code IS NOT NULL AND customs_code <> '') AS customs_code
  FROM declarations
), c AS (
  SELECT 'cargo' AS table_name, COUNT(*) AS n,
         SUM(direction IS NOT NULL AND direction <> '') AS direction,
         SUM(operation_code IS NOT NULL AND operation_code <> '') AS operation_code,
         SUM(origin IS NOT NULL AND origin <> '') AS origin,
         SUM(hs_code IS NOT NULL AND hs_code <> '') AS hs_code,
         SUM(unit IS NOT NULL AND unit <> '') AS unit,
         SUM(tax_code IS NOT NULL) AS tax_code,
         SUM(quantity IS NOT NULL) AS quantity,
         SUM(unit_price IS NOT NULL) AS unit_price,
         SUM(value IS NOT NULL) AS value
  FROM cargo
)
SELECT * FROM d
UNION ALL BY NAME
SELECT * FROM c
''')
field_coverage

,table_name,n,direction,operation_code,export_country,import_country,delivery_term,currency,port_code,customs_code,origin,hs_code,unit,tax_code,quantity,unit_price,value
0,declarations,563919,563909.0,563910.0,563788.0,563441.0,563909.0,563908.0,563911.0,563919.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,cargo,2012434,2012397.0,2012425.0,NaN,NaN,NaN,NaN,NaN,NaN,2012147.0,2012414.0,2012417.0,2012434.0,2012427.0,2010524.0,2012209.0


## 3. Rule mining 2 chiều — declaration level
Các rule thống trị `A -> B`: chỉ lấy consequent phổ biến nhất cho mỗi antecedent.

In [4]:
declaration_pairs = [
    (['operation_code'], 'direction'),
    (['operation_code'], 'import_country'),
    (['operation_code'], 'export_country'),
    (['operation_code'], 'currency'),
    (['operation_code'], 'delivery_term'),
    (['operation_code'], 'port_code'),
    (['export_country'], 'currency'),
    (['import_country'], 'currency'),
    (['port_code'], 'customs_code'),
    (['customs_code'], 'port_code'),
]

all_decl_rules = []
for ants, target in declaration_pairs:
    df = dominant_rules(con, 'declarations', ants, target, min_support=100, min_confidence=0.70, limit=500)
    df.insert(0, 'rule_family', f"{'+'.join(ants)}->{target}")
    all_decl_rules.append(df)

decl_rules = pd.concat(all_decl_rules, ignore_index=True)
save(decl_rules, 'declaration_2way_rules.csv')
save(decl_rules, 'declaration_2way_rules.json')
decl_rules.head(30)

,rule_family,operation_code,direction,rule_count,antecedent_count,confidence,rn,action,import_country,export_country,currency,delivery_term,port_code,customs_code
0,operation_code->direction,E11,N,106143,106143,1.000000,1,AUTOFILL,NaN,NaN,NaN,NaN,NaN,NaN
1,operation_code->direction,E15,N,51313,51313,1.000000,1,AUTOFILL,NaN,NaN,NaN,NaN,NaN,NaN
2,operation_code->direction,A12,N,36421,36421,1.000000,1,AUTOFILL,NaN,NaN,NaN,NaN,NaN,NaN
3,operation_code->direction,E13,N,27703,27703,1.000000,1,AUTOFILL,NaN,NaN,NaN,NaN,NaN,NaN
4,operation_code->direction,A41,N,15959,15959,1.000000,1,AUTOFILL,NaN,NaN,NaN,NaN,NaN,NaN
5,operation_code->direction,E31,N,13556,13556,1.000000,1,AUTOFILL,NaN,NaN,NaN,NaN,NaN,NaN
6,operation_code->direction,E21,N,5688,5688,1.000000,1,AUTOFILL,NaN,NaN,NaN,NaN,NaN,NaN
7,operation_code->direction,E52,X,3777,3777,1.000000,1,AUTOFILL,NaN,NaN,NaN,NaN,NaN,NaN
8,operation_code->direction,B13,X,3406,3406,1.000000,1,AUTOFILL,NaN,NaN,NaN,NaN,NaN,NaN
9,operation_code->direction,H11,N,2696,2696,1.000000,1,AUTOFILL,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
# Candidate AUTOFILL/SUGGEST distribution
rule_counts = decl_rules.groupby(['rule_family', 'action']).size().reset_index(name='rules')
rule_counts.pivot(index='rule_family', columns='action', values='rules').fillna(0).astype(int)

action,AUTOFILL,HINT,SUGGEST
rule_family,,,
customs_code->port_code,4,2,5
export_country->currency,6,5,15
import_country->currency,47,1,15
operation_code->currency,0,2,12
operation_code->delivery_term,1,0,2
operation_code->direction,24,0,0
operation_code->export_country,9,0,2
operation_code->import_country,15,1,0
operation_code->port_code,1,2,1


## 4. Rule mining 3 chiều — declaration level
Tìm rule `A+B -> C`, rất hữu ích cho AI vì giảm nhập tay khi một field đơn lẻ chưa đủ tin cậy.

In [6]:
declaration_3way = [
    (['operation_code', 'export_country'], 'currency'),
    (['operation_code', 'import_country'], 'currency'),
    (['operation_code', 'delivery_term'], 'port_code'),
    (['operation_code', 'port_code'], 'customs_code'),
    (['direction', 'export_country'], 'currency'),
    (['direction', 'port_code'], 'customs_code'),
]

all_3way = []
for ants, target in declaration_3way:
    df = dominant_rules(con, 'declarations', ants, target, min_support=50, min_confidence=0.80, limit=1000)
    df.insert(0, 'rule_family', f"{'+'.join(ants)}->{target}")
    all_3way.append(df)

decl_3way_rules = pd.concat(all_3way, ignore_index=True)
save(decl_3way_rules, 'declaration_3way_rules.csv')
save(decl_3way_rules, 'declaration_3way_rules.json')
decl_3way_rules.head(40)

,rule_family,operation_code,export_country,currency,rule_count,antecedent_count,confidence,rn,action,import_country,delivery_term,port_code,customs_code,direction
0,operation_code+export_country->currency,E11,PH,USD,2879,2879,1.0,1,AUTOFILL,NaN,NaN,NaN,NaN,NaN
1,operation_code+export_country->currency,E31,HK,USD,1705,1705,1.0,1,AUTOFILL,NaN,NaN,NaN,NaN,NaN
2,operation_code+export_country->currency,A41,US,USD,542,542,1.0,1,AUTOFILL,NaN,NaN,NaN,NaN,NaN
3,operation_code+export_country->currency,A41,MY,USD,469,469,1.0,1,AUTOFILL,NaN,NaN,NaN,NaN,NaN
4,operation_code+export_country->currency,E31,KR,USD,400,400,1.0,1,AUTOFILL,NaN,NaN,NaN,NaN,NaN
5,operation_code+export_country->currency,A12,NZ,USD,397,397,1.0,1,AUTOFILL,NaN,NaN,NaN,NaN,NaN
6,operation_code+export_country->currency,E15,HK,USD,321,321,1.0,1,AUTOFILL,NaN,NaN,NaN,NaN,NaN
7,operation_code+export_country->currency,E13,KR,USD,297,297,1.0,1,AUTOFILL,NaN,NaN,NaN,NaN,NaN
8,operation_code+export_country->currency,A11,CN,USD,288,288,1.0,1,AUTOFILL,NaN,NaN,NaN,NaN,NaN
9,operation_code+export_country->currency,G13,CN,USD,287,287,1.0,1,AUTOFILL,NaN,NaN,NaN,NaN,NaN


## 5. HS hierarchy mining — cargo level
Phân tích theo HS2/HS4/HS6 để tìm unit/tax_code/origin có thể autofill hoặc suggest.

In [7]:
hs_rule_specs = [
    (['hs2'], 'unit', 500, 0.70),
    (['hs4'], 'unit', 100, 0.80),
    (['hs6'], 'unit', 50, 0.85),
    (['direction', 'hs2'], 'tax_code', 500, 0.70),
    (['direction', 'hs4'], 'tax_code', 100, 0.80),
    (['direction', 'hs6'], 'tax_code', 50, 0.85),
    (['operation_code', 'hs2'], 'tax_code', 500, 0.70),
    (['operation_code', 'hs4'], 'tax_code', 100, 0.80),
    (['operation_code', 'hs6'], 'tax_code', 50, 0.85),
    (['operation_code'], 'origin', 500, 0.70),
    (['operation_code', 'hs2'], 'origin', 200, 0.75),
]

all_hs_rules = []
for ants, target, support, conf in hs_rule_specs:
    df = dominant_rules(con, 'cargo', ants, target, min_support=support, min_confidence=conf, limit=1500)
    df.insert(0, 'rule_family', f"{'+'.join(ants)}->{target}")
    all_hs_rules.append(df)

hs_rules = pd.concat(all_hs_rules, ignore_index=True)
save(hs_rules, 'cargo_hs_rules.csv')
save(hs_rules, 'cargo_hs_rules.json')
hs_rules.head(50)

,rule_family,hs2,unit,rule_count,antecedent_count,confidence,rn,action,hs4,hs6,direction,tax_code,operation_code,origin
0,hs2->unit,87,PCE,8274,8331,0.993158,1,AUTOFILL,NaN,NaN,NaN,NaN,NaN,NaN
1,hs2->unit,41,FTK,16086,16245,0.990212,1,AUTOFILL,NaN,NaN,NaN,NaN,NaN,NaN
2,hs2->unit,62,PCE,14290,14443,0.989407,1,SUGGEST,NaN,NaN,NaN,NaN,NaN,NaN
3,hs2->unit,90,PCE,175128,177305,0.987722,1,SUGGEST,NaN,NaN,NaN,NaN,NaN,NaN
4,hs2->unit,42,PCE,3744,3813,0.981904,1,SUGGEST,NaN,NaN,NaN,NaN,NaN,NaN
5,hs2->unit,64,PR,29989,30843,0.972311,1,SUGGEST,NaN,NaN,NaN,NaN,NaN,NaN
6,hs2->unit,61,PCE,231946,240062,0.966192,1,SUGGEST,NaN,NaN,NaN,NaN,NaN,NaN
7,hs2->unit,81,PCE,4118,4272,0.963951,1,SUGGEST,NaN,NaN,NaN,NaN,NaN,NaN
8,hs2->unit,85,PCE,288273,305262,0.944346,1,SUGGEST,NaN,NaN,NaN,NaN,NaN,NaN
9,hs2->unit,60,MTK,51507,54656,0.942385,1,SUGGEST,NaN,NaN,NaN,NaN,NaN,NaN


## 6. Rule novelty: ưu tiên rule mới có impact cao
Danh sách này giúp review: nhiều dòng + confidence cao + chưa chắc đã có trong engine hiện tại.

In [8]:
impact_rules = pd.concat([
    decl_rules.assign(level='declaration_2way'),
    decl_3way_rules.assign(level='declaration_3way'),
    hs_rules.assign(level='cargo_hs'),
], ignore_index=True, sort=False)

impact_rules['impact_score'] = impact_rules['antecedent_count'] * impact_rules['confidence']
impact_rules = impact_rules.sort_values(['action', 'impact_score'], ascending=[True, False])

autofill_candidates = impact_rules.query("action == 'AUTOFILL'").sort_values('impact_score', ascending=False)
suggest_candidates = impact_rules.query("action == 'SUGGEST'").sort_values('impact_score', ascending=False)

save(autofill_candidates.head(500), 'candidate_autofill_rules.csv')
save(suggest_candidates.head(500), 'candidate_suggest_rules.csv')
autofill_candidates.head(50)

,rule_family,operation_code,direction,rule_count,antecedent_count,confidence,rn,action,import_country,export_country,currency,delivery_term,port_code,customs_code,level,hs2,unit,hs4,hs6,tax_code,origin,impact_score
4186,operation_code->origin,E42,NaN,740354,741090,0.999007,1,AUTOFILL,NaN,NaN,NaN,NaN,NaN,NaN,cargo_hs,NaN,NaN,NaN,NaN,NaN,VN,740354.0
4229,operation_code+hs2->origin,E42,NaN,233917,233920,0.999987,1,AUTOFILL,NaN,NaN,NaN,NaN,NaN,NaN,cargo_hs,61,NaN,NaN,NaN,NaN,VN,233917.0
21,operation_code->direction,E42,X,210414,210415,0.999995,1,AUTOFILL,NaN,NaN,NaN,NaN,NaN,NaN,declaration_2way,NaN,NaN,NaN,NaN,NaN,NaN,210414.0
46,operation_code->export_country,E42,NaN,210413,210414,0.999995,1,AUTOFILL,NaN,VN,NaN,NaN,NaN,NaN,declaration_2way,NaN,NaN,NaN,NaN,NaN,NaN,210413.0
709,hs4->unit,NaN,NaN,162569,162569,1.000000,1,AUTOFILL,NaN,NaN,NaN,NaN,NaN,NaN,cargo_hs,NaN,PCE,9021,NaN,NaN,NaN,162569.0
4231,operation_code+hs2->origin,E42,NaN,155361,155367,0.999961,1,AUTOFILL,NaN,NaN,NaN,NaN,NaN,NaN,cargo_hs,90,NaN,NaN,NaN,NaN,VN,155361.0
945,hs6->unit,NaN,NaN,110474,110474,1.000000,1,AUTOFILL,NaN,NaN,NaN,NaN,NaN,NaN,cargo_hs,NaN,PCE,NaN,902140,NaN,NaN,110474.0
0,operation_code->direction,E11,N,106143,106143,1.000000,1,AUTOFILL,NaN,NaN,NaN,NaN,NaN,NaN,declaration_2way,NaN,NaN,NaN,NaN,NaN,NaN,106143.0
24,operation_code->import_country,E11,NaN,106143,106143,1.000000,1,AUTOFILL,VN,NaN,NaN,NaN,NaN,NaN,declaration_2way,NaN,NaN,NaN,NaN,NaN,NaN,106143.0
4247,operation_code+hs2->origin,E42,NaN,96574,96903,0.996605,1,AUTOFILL,NaN,NaN,NaN,NaN,NaN,NaN,cargo_hs,85,NaN,NaN,NaN,NaN,VN,96574.0


## 7. Price anomaly baseline theo HS6
Không phải rule autofill, nhưng rất hữu ích cho AI guardrail: cảnh báo unit_price lạ theo HS6 + unit.

In [9]:
price_baseline = q(con, '''
WITH clean AS (
  SELECT hs6, unit, unit_price, value, quantity
  FROM cargo
  WHERE hs6 IS NOT NULL AND hs6 <> ''
    AND unit IS NOT NULL AND unit <> ''
    AND unit_price IS NOT NULL AND unit_price > 0
), stats AS (
  SELECT hs6, unit,
         COUNT(*) AS n,
         quantile_cont(unit_price, 0.10) AS p10,
         quantile_cont(unit_price, 0.25) AS p25,
         quantile_cont(unit_price, 0.50) AS median,
         quantile_cont(unit_price, 0.75) AS p75,
         quantile_cont(unit_price, 0.90) AS p90,
         quantile_cont(unit_price, 0.99) AS p99,
         AVG(unit_price) AS avg_price
  FROM clean
  GROUP BY hs6, unit
)
SELECT *, p75 - p25 AS iqr,
       GREATEST(p10 / NULLIF(median, 0), 0) AS low_ratio,
       p90 / NULLIF(median, 0) AS high_ratio
FROM stats
WHERE n >= 50
ORDER BY n DESC
''')
save(price_baseline, 'hs6_unit_price_baseline.csv')
price_baseline.head(30)

,hs6,unit,n,p10,p25,median,p75,p90,p99,avg_price,iqr,low_ratio,high_ratio
0,392690,PCE,131849,0.050000,0.230000,1.360000,5.180000,1.135000e+01,2.250000e+05,2.629826e+04,4.950000,0.036765,8.345588
1,902140,PCE,110474,31.820000,37.060000,42.730000,49.950000,5.574000e+01,7.154000e+01,4.422719e+01,12.890000,0.744676,1.304470
2,482110,PCE,78043,0.006795,0.009720,0.023380,0.074960,3.680000e+02,3.300000e+03,3.080759e+02,0.065240,0.290650,15739.948674
3,580710,PCE,63302,0.011960,0.019000,0.039370,0.086700,5.690000e+02,2.027000e+03,1.543011e+02,0.067700,0.303785,14452.628905
4,902190,PCE,51887,0.200000,1.130000,4.050000,7.960000,3.626000e+01,8.601000e+01,1.112364e+01,6.830000,0.049383,8.953086
5,610463,PCE,49224,7.010000,9.490000,12.760000,15.130000,1.981000e+01,2.925000e+01,2.555864e+01,5.640000,0.549373,1.552508
6,611030,PCE,49091,4.070000,6.930000,10.170000,14.600000,2.212000e+01,5.060000e+01,1.811471e+02,7.670000,0.400197,2.175025
7,610990,PCE,48245,3.540000,4.260000,6.950000,9.010000,1.177000e+01,3.430480e+01,1.069938e+02,4.750000,0.509353,1.693525
8,851830,PCE,48200,3.720000,4.580000,8.280000,13.030000,1.424000e+01,1.569000e+01,6.385317e+02,8.450000,0.449275,1.719807
9,732690,PCE,41708,0.084000,1.000000,149.000000,14000.000000,1.150000e+06,1.756800e+07,1.167006e+06,13999.000000,0.000564,7718.120805


## 8. Export manifest cho engine
JSON gọn để kỹ sư có thể review rồi đưa một phần vào `scripts/autofill_engine.py`.

In [10]:
manifest = {
    'thresholds': {'AUTOFILL': 0.99, 'SUGGEST': 0.80, 'HINT': 0.70},
    'outputs': {
        'declaration_2way_rules': str(OUTPUT_DIR / 'declaration_2way_rules.csv'),
        'declaration_3way_rules': str(OUTPUT_DIR / 'declaration_3way_rules.csv'),
        'cargo_hs_rules': str(OUTPUT_DIR / 'cargo_hs_rules.csv'),
        'candidate_autofill_rules': str(OUTPUT_DIR / 'candidate_autofill_rules.csv'),
        'candidate_suggest_rules': str(OUTPUT_DIR / 'candidate_suggest_rules.csv'),
        'hs6_unit_price_baseline': str(OUTPUT_DIR / 'hs6_unit_price_baseline.csv'),
    },
    'counts': {
        'declaration_2way_rules': int(len(decl_rules)),
        'declaration_3way_rules': int(len(decl_3way_rules)),
        'cargo_hs_rules': int(len(hs_rules)),
        'autofill_candidates': int(len(autofill_candidates)),
        'suggest_candidates': int(len(suggest_candidates)),
    }
}
import json
with open(OUTPUT_DIR / 'rule_mining_manifest.json', 'w', encoding='utf-8') as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)
manifest

{'thresholds': {'AUTOFILL': 0.99, 'SUGGEST': 0.8, 'HINT': 0.7},
 'outputs': {'declaration_2way_rules': '/Volumes/FastVault-Apps/Projects/ai-vvmv/notebooks/outputs/declaration_2way_rules.csv',
  'declaration_3way_rules': '/Volumes/FastVault-Apps/Projects/ai-vvmv/notebooks/outputs/declaration_3way_rules.csv',
  'cargo_hs_rules': '/Volumes/FastVault-Apps/Projects/ai-vvmv/notebooks/outputs/cargo_hs_rules.csv',
  'candidate_autofill_rules': '/Volumes/FastVault-Apps/Projects/ai-vvmv/notebooks/outputs/candidate_autofill_rules.csv',
  'candidate_suggest_rules': '/Volumes/FastVault-Apps/Projects/ai-vvmv/notebooks/outputs/candidate_suggest_rules.csv',
  'hs6_unit_price_baseline': '/Volumes/FastVault-Apps/Projects/ai-vvmv/notebooks/outputs/hs6_unit_price_baseline.csv'},
 'counts': {'declaration_2way_rules': 202,
  'declaration_3way_rules': 477,
  'cargo_hs_rules': 3659,
  'autofill_candidates': 2640,
  'suggest_candidates': 1650}}